> 在线讲义：[第 55 章 · SLAM 的优化：重投影误差与 BA、稀疏 Hessian 与 Schur 补、鲁棒核、边缘化](https://wikieden.github.io/math-and-eval-stats/lessons/55-slam-optimization.html) · 本书《数学与实验统计》

# 第 55 章 · SLAM 的优化：重投影误差与 BA、稀疏 Hessian 与 Schur 补、鲁棒核、边缘化

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/55-slam-optimization.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 重投影 | π(K(Rp + t)) − u | `reproj_residual`、`numeric_jac` | (−2.5, 1.0)；2×3 雅可比 |
| 第二步 Schur | 规模统计 | `ba_sizes(10, 1000, 5)` | 3060 变量，60×60 |
| 第三步 鲁棒核 | IRLS | `line_fit("cauchy")` | 斜率 1.9907 |
| 第四步 边缘化 | Schur 补 | `marginalize()` | 2.6667 |

例子：一个 3D 点在一台相机里的重投影误差和雅可比的形状；10 台相机、1000 个点的 BA 的 Hessian 规模与 Schur 补；
带一个外点的直线拟合上 Huber / Cauchy 鲁棒核（IRLS）；两个变量的高斯里边缘化掉一个（信息矩阵的 Schur 补）。
里程碑 ME-M33 的“用 scipy.optimize.least_squares 做小型 BA、看稀疏 Jacobian”留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
# 下面几段来自 ch52_camera_geometry.py（第 52 章）
import numpy as np
K = np.array([[500.0, 0, 320], [0, 500.0, 240], [0, 0, 1]])

**函数 `reproj_residual`**：重投影误差 e = π(K(R p + t)) − u_obs（2 维）。

In [ ]:
def reproj_residual(p_world: np.ndarray, uv_obs: np.ndarray, r: np.ndarray, t: np.ndarray) -> np.ndarray:
    """重投影误差 e = π(K(R p + t)) − u_obs（2 维）。"""
    pc = r @ p_world + t
    uvw = K @ pc
    return uvw[:2] / uvw[2] - uv_obs

**函数 `numeric_jac`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def numeric_jac(f, x: np.ndarray, h: float = 1e-6) -> np.ndarray:
    fx = f(x)
    return np.stack([(f(x + h * e) - fx) / h for e in np.eye(len(x))], 1)

**函数 `ba_sizes`**：BA：每台相机 6 个参数、每个点 3 个。Hessian 是 (6c + 3p)²；相机-点块只在“点被这台相机看到”时非零。

In [ ]:
def ba_sizes(ncam: int, npt: int, obs_per_pt: int) -> dict:
    """BA：每台相机 6 个参数、每个点 3 个。Hessian 是 (6c + 3p)²；相机-点块只在“点被这台相机看到”时非零。
    相机-相机、点-点之间的非对角块都是 0。Schur 补把点消掉，只解 6c × 6c 的“缩减相机系统”。"""
    n = 6 * ncam + 3 * npt
    nnz = 36 * ncam + 9 * npt + 2 * 18 * npt * obs_per_pt   # 相机对角块 + 点对角块 + 相机-点块（上下对称两份）
    return {"变量数": n, "满矩阵元素": n * n, "非零元（约）": nnz, "缩减系统": (6 * ncam) ** 2}

**函数 `robust_weights`**：IRLS 的权重 w(r) = ρ'(r)/r：Huber 在 |r| > δ 时 δ/|r|；Cauchy 是 1/(1 + (r/δ)²)。

In [ ]:
def robust_weights(r: np.ndarray, delta: float = 1.0) -> tuple[np.ndarray, np.ndarray]:
    """IRLS 的权重 w(r) = ρ'(r)/r：Huber 在 |r| > δ 时 δ/|r|；Cauchy 是 1/(1 + (r/δ)²)。"""
    a = np.abs(r)
    huber = np.where(a <= delta, 1.0, delta / a)
    cauchy = 1.0 / (1.0 + (r / delta) ** 2)
    return huber, cauchy

**函数 `line_fit`**：y = 2x + 1 上 10 个点加小噪声，再加一个外点 (9, 0)。最小二乘 vs IRLS（Huber / Cauchy）。返回 (斜率, 截距)。

In [ ]:
def line_fit(kind: str, iters: int = 20) -> tuple[float, float]:
    """y = 2x + 1 上 10 个点加小噪声，再加一个外点 (9, 0)。最小二乘 vs IRLS（Huber / Cauchy）。返回 (斜率, 截距)。"""
    rng = np.random.default_rng(0)
    x = np.arange(10.0)
    y = 2 * x + 1 + 0.1 * rng.standard_normal(10)
    x, y = np.r_[x, 9.0], np.r_[y, 0.0]
    a = np.c_[x, np.ones_like(x)]
    w = np.ones_like(x)
    for _ in range(iters if kind != "ls" else 1):
        sol = np.linalg.solve(a.T @ (w[:, None] * a), a.T @ (w * y))
        r = y - a @ sol
        if kind == "ls":
            break
        h, c = robust_weights(r, 1.0)
        w = h if kind == "huber" else c
    return float(sol[0]), float(sol[1])

**函数 `marginalize`**：两个变量的高斯，信息矩阵 Λ = [[a, b], [b, c]]。边缘化掉 x1：x0 的信息 = a − b²/c（Schur 补），

In [ ]:
def marginalize() -> tuple[np.ndarray, float]:
    """两个变量的高斯，信息矩阵 Λ = [[a, b], [b, c]]。边缘化掉 x1：x0 的信息 = a − b²/c（Schur 补），
    和直接从协方差 Σ = Λ⁻¹ 取 Σ00 再求逆一样。返回 (Λ, Schur 补)。"""
    lam = np.array([[4.0, -2.0], [-2.0, 3.0]])
    return lam, lam[0, 0] - lam[0, 1] ** 2 / lam[1, 1]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 重投影误差和雅可比的形状")
p = np.array([0.5, -0.2, 4.0])
r, t = np.eye(3), np.zeros(3)
obs = np.array([385.0, 214.0])
e = reproj_residual(p, obs, r, t)
jp = numeric_jac(lambda q: reproj_residual(q, obs, r, t), p)
print(f"  点 {p.tolist()}、观测 {obs.tolist()}：误差 {np.round(e, 3).tolist()} 像素")
print(f"  对点的雅可比（2×3）：{np.round(jp, 2).tolist()}；对相机位姿的雅可比是 2×6")

In [ ]:
print("\n第二步 BA 的 Hessian：稀疏 + Schur 补")
s = ba_sizes(10, 1000, 5)
for k, v in s.items():
    print(f"  {k}：{v:,}")

In [ ]:
print("\n第三步 鲁棒核：外点的权重被压下去")
rr = np.array([0.5, 2.0, 10.0])
h, c = robust_weights(rr)
print(f"  残差 {rr.tolist()}（δ = 1）：Huber 权重 {np.round(h, 3).tolist()}，Cauchy 权重 {np.round(c, 3).tolist()}")
for k in ("ls", "huber", "cauchy"):
    print(f"  直线 y = 2x + 1 加一个外点，{k}：斜率 {line_fit(k)[0]:.4f}、截距 {line_fit(k)[1]:.4f}")

In [ ]:
print("\n第四步 边缘化 = 信息矩阵的 Schur 补")
lam, sc = marginalize()
print(f"  Λ = {lam.tolist()}：边缘化 x1 后 x0 的信息 {sc:.4f}；从协方差算 1/Σ00 = {1 / np.linalg.inv(lam)[0, 0]:.4f}")

## 3. 代码里才看得到的

1. **雅可比用数值差分**：h = 1e-6 的前向差分，精度约 1e-4；真实 BA 用解析雅可比（对位姿的部分在切空间里求，第 19 章）。
2. **Hessian 的非零元是估算**：只数了块的大小，没有真的构造矩阵；用 `scipy.optimize.least_squares` 做一个小型 BA 并画出稀疏 Jacobian，留给自己做（里程碑 ME-M33）。
3. **IRLS 的权重用上一轮的残差**：第一轮所有权重为 1（就是最小二乘），之后逐轮更新；Cauchy 从最小二乘的解出发，这个例子里能收敛到内点上，外点更多时可能陷进局部最优。

## 4. 自己改一改

1. 把外点改成 (9, −20)，Huber 和 Cauchy 的斜率各变成多少？
2. 在 `marginalize` 里加第三个变量（3×3 的 Λ），边缘化中间那个，看另外两个变量之间是否出现新的关联（填充）。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> BA 投回照片比位置，每个残差只连一相机一个点；
> Hessian 像箭头，Schur 补消点，只解小相机系统；
> 一个外点能拉偏全局，鲁棒核降权，Huber 温和 Cauchy 狠；
> 扔旧帧要边缘化，Schur 补折成先验，代价是填充。